In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# Download and extract the edges2shoes dataset
!wget http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/edges2shoes.tar.gz
!tar -xf edges2shoes.tar.gz

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

--2026-03-11 06:02:50--  http://efrosgans.eecs.berkeley.edu/pix2pix/datasets/edges2shoes.tar.gz
Resolving efrosgans.eecs.berkeley.edu (efrosgans.eecs.berkeley.edu)... 128.32.244.190
Connecting to efrosgans.eecs.berkeley.edu (efrosgans.eecs.berkeley.edu)|128.32.244.190|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2165283376 (2.0G) [application/x-gzip]
Saving to: ‘edges2shoes.tar.gz’

edges2shoes.tar.gz  100%[===================>]   2.02G  7.33MB/s    in 7m 5s   

2026-03-11 06:09:54 (4.86 MB/s) - ‘edges2shoes.tar.gz’ saved [2165283376/2165283376]

Using device: cuda


In [5]:
class Edges2ShoesDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        self.image_files = [f for f in os.listdir(root_dir) if f.endswith('.jpg')]
        # Transform: convert to tensor and normalize to [-1, 1]
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(img_path).convert('RGB')
        
        # Split the concatenated image in half
        w, h = image.size
        w_half = w // 2
        
        # Left half is the input (edges), right half is the target (real shoe)
        input_image = image.crop((0, 0, w_half, h))
        target_image = image.crop((w_half, 0, w, h))
        
        input_tensor = self.transform(input_image)
        target_tensor = self.transform(target_image)
        
        return input_tensor, target_tensor

# Load a small batch for faster Colab testing
train_dataset = Edges2ShoesDataset('edges2shoes/val') # Using 'val' folder for a quicker epoch during testing
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

In [6]:
class PatchGANDiscriminator(nn.Module):
    def __init__(self):
        super(PatchGANDiscriminator, self).__init__()
        
        # It takes 6 channels because we concatenate the Input (Edges) and the Target/Generated image [cite: 706]
        self.model = nn.Sequential(
            nn.Conv2d(6, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Stride of 1 to slow down spatial reduction [cite: 710]
            nn.Conv2d(256, 512, kernel_size=4, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Final output layer (produces the patch-level map)
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1),
            nn.Sigmoid() # Outputs probabilities [0, 1] [cite: 705]
        )

    def forward(self, input_img, target_img):
        # Concatenate condition (edges) and image (real or generated) [cite: 282]
        merged = torch.cat([input_img, target_img], dim=1)
        return self.model(merged)

discriminator = PatchGANDiscriminator().to(device)

In [10]:
generator = Generator().to(device)
discriminator = Discriminator().to(device)

# Losses
criterion_GAN = nn.BCELoss() # Adversarial loss [cite: 314]
criterion_L1 = nn.L1Loss()   # Reconstruction loss [cite: 325]

# Optimizers (Adam with LR 0.0002) [cite: 46, 47]
optimizer_G = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

lambda_L1 = 100 # Weight for the L1 Loss [cite: 339]

num_epochs = 5 # Set low for initial Colab testing

for epoch in range(num_epochs):
    for i, (edges, real_shoes) in enumerate(train_loader):
        edges = edges.to(device)
        real_shoes = real_shoes.to(device)
        
        # ---------------------
        #  Train Discriminator
        # ---------------------
        optimizer_D.zero_grad()
        
        # Generate fake shoe
        fake_shoes = generator(edges)
        
        # Classify Real
        pred_real = discriminator(edges, real_shoes)
        loss_D_real = criterion_GAN(pred_real, torch.ones_like(pred_real))
        
        # Classify Fake (detach so we don't backprop through Generator yet)
        pred_fake = discriminator(edges, fake_shoes.detach())
        loss_D_fake = criterion_GAN(pred_fake, torch.zeros_like(pred_fake))
        
        # Total D Loss [cite: 318, 321]
        loss_D = (loss_D_real + loss_D_fake) * 0.5
        loss_D.backward()
        optimizer_D.step()

        # -----------------
        #  Train Generator
        # -----------------
        optimizer_G.zero_grad()
        
        # Discriminator evaluates the generated fake shoe again (after D is updated)
        pred_fake_for_G = discriminator(edges, fake_shoes)
        
        # Generator wants to fool Discriminator (aim for label 1) [cite: 741]
        loss_G_GAN = criterion_GAN(pred_fake_for_G, torch.ones_like(pred_fake_for_G))
        
        # Generator also wants to match the exact pixels of the real shoe [cite: 326]
        loss_G_L1 = criterion_L1(fake_shoes, real_shoes)
        
        # Total G Loss [cite: 332]
        loss_G = loss_G_GAN + (lambda_L1 * loss_G_L1)
        loss_G.backward()
        optimizer_G.step()
        
    print(f"Epoch [{epoch+1}/{num_epochs}] | D Loss: {loss_D.item():.4f} | G Loss: {loss_G.item():.4f}")

# Visualization of the last batch
def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.detach().cpu().numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.title("Input Edges")
imshow(edges[0])

plt.subplot(1, 3, 2)
plt.title("Generated Shoe")
imshow(fake_shoes[0])

plt.subplot(1, 3, 3)
plt.title("Ground Truth Shoe")
imshow(real_shoes[0])
plt.show()

NameError: name 'Discriminator' is not defined

In [9]:
import torch
import torch.nn as nn

class UNetDown(nn.Module):
    def __init__(self, in_channels, out_channels, normalize=True, dropout=0.0):
        super(UNetDown, self).__init__()
        layers = [nn.Conv2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False)]
        if normalize:
            layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.LeakyReLU(0.2))
        if dropout > 0.0:
            layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

class UNetUp(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.0):
        super(UNetUp, self).__init__()
        layers = [
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        ]
        if dropout > 0.0:
            layers.append(nn.Dropout(dropout))
        self.model = nn.Sequential(*layers)

    def forward(self, x, skip_input):
        x = self.model(x)
        # Concatenate with skip connection
        x = torch.cat((x, skip_input), 1)
        return x

class Generator(nn.Module):
    def __init__(self, in_channels=3, out_channels=3):
        super(Generator, self).__init__()
        
        # Downsampling
        self.down1 = UNetDown(in_channels, 64, normalize=False)
        self.down2 = UNetDown(64, 128)
        self.down3 = UNetDown(128, 256)
        self.down4 = UNetDown(256, 512, dropout=0.5)
        self.down5 = UNetDown(512, 512, dropout=0.5)
        self.down6 = UNetDown(512, 512, dropout=0.5)
        self.down7 = UNetDown(512, 512, dropout=0.5)
        self.down8 = UNetDown(512, 512, normalize=False, dropout=0.5)
        
        # Upsampling
        self.up1 = UNetUp(512, 512, dropout=0.5)
        self.up2 = UNetUp(1024, 512, dropout=0.5)
        self.up3 = UNetUp(1024, 512, dropout=0.5)
        self.up4 = UNetUp(1024, 512, dropout=0.5)
        self.up5 = UNetUp(1024, 256)
        self.up6 = UNetUp(512, 128)
        self.up7 = UNetUp(256, 64)

        self.final_up = nn.Sequential(
            nn.Upsample(scale_factor=2),
            nn.ZeroPad2d((1, 0, 1, 0)),
            nn.Conv2d(128, out_channels, kernel_size=4, padding=1),
            nn.Tanh() # Output values between -1 and 1
        )

    def forward(self, x):
        d1 = self.down1(x)
        d2 = self.down2(d1)
        d3 = self.down3(d2)
        d4 = self.down4(d3)
        d5 = self.down5(d4)
        d6 = self.down6(d5)
        d7 = self.down7(d6)
        d8 = self.down8(d7)
        
        u1 = self.up1(d8, d7)
        u2 = self.up2(u1, d6)
        u3 = self.up3(u2, d5)
        u4 = self.up4(u3, d4)
        u5 = self.up5(u4, d3)
        u6 = self.up6(u5, d2)
        u7 = self.up7(u6, d1)
        
        return self.final_up(u7)
